# Scaling Laws for Neural Language Models

## Learning Objectives

1. Understand power law scaling relationships for neural networks
2. Fit and extrapolate scaling laws from empirical measurements
3. Allocate compute optimally between model and data
4. Predict model performance before training
5. Analyze scaling laws for different factors (size, data, compute)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)

print("Scaling Laws for Language Models")
print("=================================")

## Level 1: Power Law Basics

Loss follows: L(N) = a * N^(-alpha)

In [ ]:
# Empirical measurements from literature
# Model size (parameters) vs validation loss
model_sizes = np.array([125e6, 350e6, 1.3e9, 6.7e9, 13e9, 175e9])  # Parameters
losses = np.array([4.2, 3.95, 3.5, 2.9, 2.5, 2.1])  # Validation loss

# Define power law function
def power_law(N, a, alpha):
    return a * np.power(N, -alpha)

# Fit power law to data
popt, pcov = curve_fit(power_law, model_sizes, losses, p0=[10, 0.08])
a, alpha = popt

print(f"Fitted power law: L(N) = {a:.3f} * N^(-{alpha:.3f})")
print(f"Alpha (exponent): {alpha:.4f}")
print(f"Meaning: 10x bigger model → {(10**alpha):.2f}x better loss reduction")

# Verify fit
predicted_losses = power_law(model_sizes, a, alpha)
mse = np.mean((losses - predicted_losses) ** 2)
print(f"Fit quality (MSE): {mse:.6f}")

# Plot
fig, ax = plt.subplots(figsize=(10, 6))
ax.loglog(model_sizes, losses, 'o-', label='Measured', markersize=10, linewidth=2)

# Plot fitted curve
x_fit = np.logspace(8, 11.5, 100)
y_fit = power_law(x_fit, a, alpha)
ax.loglog(x_fit, y_fit, '--', label=f'Fitted: L = {a:.2f}*N^(-{alpha:.3f})', linewidth=2)

ax.set_xlabel('Model Size (parameters)', fontsize=12)
ax.set_ylabel('Validation Loss', fontsize=12)
ax.set_title('Power Law Scaling for Language Models', fontsize=14)
ax.grid(True, alpha=0.3)
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

## Level 2: Multi-Factor Scaling and Compute Allocation

Scale parameters, data, and compute optimally

In [ ]:
# Scaling laws for different factors
# L(N) = a_N * N^(-0.07)
# L(D) = a_D * D^(-0.09)
# L(C) = a_C * C^(-0.08)
# where C = 6*N*D (total compute in FLOPs)

# Given a compute budget, what's the optimal allocation?
def compute_optimal_allocation(compute_budget):
    """
    Optimal allocation: equal FLOPs for parameters and data
    C = 6*N*D
    Optimal: N^opt ∝ C^(1/1.63), D^opt ∝ C^(0.63/1.63)
    Approximately: D ≈ 20*N
    """
    # Compute-optimal ratio
    N_opt = np.sqrt(compute_budget / 6)
    D_opt = compute_budget / (6 * N_opt)
    
    return N_opt, D_opt

# Example compute budgets
compute_budgets = np.array([
    1e17,   # 100 petaflops
    1e18,   # 1 exaflop
    1e19,   # 10 exaflops
    1e20    # 100 exaflops
])

print("Compute-Optimal Allocation:")
print("="*70)
print(f"{'Compute Budget':<20} {'Model Size':<20} {'Tokens':<20}")
print("="*70)

allocations = []
for C in compute_budgets:
    N, D = compute_optimal_allocation(C)
    allocations.append((N, D))
    
    # Format nicely
    if C < 1e18:
        c_str = f"{C/1e15:.0f}P flops"
    else:
        c_str = f"{C/1e18:.0f}E flops"
    
    n_str = f"{N/1e9:.0f}B params"
    d_str = f"{D/1e12:.1f}T tokens"
    
    print(f"{c_str:<20} {n_str:<20} {d_str:<20}")

print("\nKey insight: For compute-optimal training, data ≈ 20x model parameters")
print("Many models are undertrained (have less data than optimal).")

## Real-World Example 1: Predict Model Performance Before Training

Use power laws to extrapolate and avoid expensive full training

In [ ]:
# Scenario: You've trained models at 10B, 30B, 100B params
# Want to predict performance at 500B without training

measured_sizes = np.array([10e9, 30e9, 100e9])
measured_losses = np.array([2.8, 2.4, 2.0])

# Fit power law
popt, _ = curve_fit(power_law, measured_sizes, measured_losses, p0=[10, 0.08])
a_pred, alpha_pred = popt

print(f"Fitted from 3 measurements: L = {a_pred:.3f} * N^(-{alpha_pred:.3f})")

# Predict at new scales
target_sizes = np.array([500e9, 1e12, 2e12])

print(f"\nPredictions (without training at these scales):")
print(f"{'Model Size':<20} {'Predicted Loss':<20} {'Improvement vs 100B':<20}")
print("-" * 60)

base_loss = power_law(100e9, a_pred, alpha_pred)

for size in target_sizes:
    pred_loss = power_law(size, a_pred, alpha_pred)
    improvement = base_loss - pred_loss
    
    size_str = f"{size/1e9:.0f}B params"
    loss_str = f"{pred_loss:.3f}"
    imp_str = f"{improvement:.3f} ({improvement/base_loss*100:.1f}%)"
    
    print(f"{size_str:<20} {loss_str:<20} {imp_str:<20}")

print(f"\nCost: 3 small training runs (10-100B) cost ~10% of one 500B run.")
print(f"Savings: 90% compute by using scaling laws for extrapolation.")

## Real-World Example 2: Scaling Laws for Different Factors

Compare how model size, data, and compute affect performance

In [ ]:
# Empirical scaling exponents
alpha_model = 0.07    # Loss ∝ N^(-0.07)  -> 10x bigger = 1.58x better
alpha_data = 0.09     # Loss ∝ D^(-0.09)  -> 10x more data = 1.79x better
alpha_compute = 0.08  # Loss ∝ C^(-0.08)  -> 10x more compute = 1.74x better

# Create range of scales
scales = np.logspace(0, 3, 100)  # 1x to 1000x

# Compute normalized loss improvements
loss_vs_model = 1.0 / (1 + scales) ** alpha_model
loss_vs_data = 1.0 / (1 + scales) ** alpha_data
loss_vs_compute = 1.0 / (1 + scales) ** alpha_compute

fig, ax = plt.subplots(figsize=(12, 6))
ax.loglog(scales, loss_vs_model, label=f'Model size (α={alpha_model})', linewidth=2)
ax.loglog(scales, loss_vs_data, label=f'Data size (α={alpha_data})', linewidth=2)
ax.loglog(scales, loss_vs_compute, label=f'Compute (α={alpha_compute})', linewidth=2)

ax.set_xlabel('Scale Factor (relative to baseline)', fontsize=12)
ax.set_ylabel('Normalized Loss', fontsize=12)
ax.set_title('Scaling Laws for Different Factors', fontsize=14)
ax.grid(True, alpha=0.3)
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

print("Observation: Data scaling slightly better than model scaling.")
print(f"But diminishing returns everywhere: 100x scale → only {1/((100)**alpha_data):.2f}x better.")

## Real-World Example 3: Optimal Allocation vs. Other Strategies

Compare compute-optimal vs. parameter-heavy vs. data-heavy strategies

In [ ]:
# Fixed compute budget: 1 exaflop (10^18 FLOPs)
compute_budget = 1e18

# Strategy 1: Compute-optimal (N ≈ D)
N_opt, D_opt = compute_optimal_allocation(compute_budget)

# Strategy 2: Parameter-heavy (larger N, less D)
N_param_heavy = 500e9  # 500B params
D_param_heavy = compute_budget / (6 * N_param_heavy)

# Strategy 3: Data-heavy (smaller N, more D)
D_data_heavy = 10e12  # 10T tokens
N_data_heavy = compute_budget / (6 * D_data_heavy)

print("Strategies for 1 Exaflop compute budget:")
print("="*70)
print(f"{'Strategy':<20} {'Model':<20} {'Data':<20} {'Predicted Loss':<15}")
print("="*70)

strategies = [
    ("Compute-optimal", N_opt, D_opt),
    ("Parameter-heavy", N_param_heavy, D_param_heavy),
    ("Data-heavy", N_data_heavy, D_data_heavy),
]

# Use fitted model from before
for name, N, D in strategies:
    # Estimate loss: combine model and data scaling
    # L ≈ L_0 * N^(-0.07) * D^(-0.09)
    loss_model_factor = (N / 1e9) ** (-0.07)
    loss_data_factor = (D / 1e12) ** (-0.09)
    loss_est = loss_model_factor * loss_data_factor * 2.0  # Normalize to ~2.0
    
    n_str = f"{N/1e9:.0f}B params"
    d_str = f"{D/1e12:.1f}T tokens"
    
    print(f"{name:<20} {n_str:<20} {d_str:<20} {loss_est:.3f}")

print("\nConclusion: Compute-optimal balances model and data.")
print("Parameter-heavy sacrifices accuracy for larger model size.")
print("Data-heavy uses more data to compensate for smaller model.")

## Key Takeaways

**Scaling Laws:**
- Loss ∝ N^(-0.07): 10x bigger = 1.6x better
- Loss ∝ D^(-0.09): 10x more data = 1.8x better
- Diminishing returns: 100x scale = only 3x better

**Compute-Optimal Allocation:**
- For budget C, allocate: N ≈ C^(0.5), D ≈ 20*N
- Don't over-scale parameters or data alone
- Balance is key: ~equal FLOPs in params and data

**Practical Applications:**
- Predict performance before training (save 90% compute)
- Allocate compute budget optimally
- Avoid undertrained models (common mistake: too many params, not enough data)

**Limitations:**
- Power laws fit in observed range (100M-175B)
- Different architectures have different constants
- Doesn't account for task-specific factors
- Breaks down at very small/large scales